# Surya OCR - One Book at a Time

Use this notebook in Google Colab. Update the config cell for one book, run the cells, then repeat for the next subject/book.

The notebook copies the PDF from Drive to `/content`, splits it into page batches, runs Surya on each batch, merges the raw results, then writes final JSON/JSONL/Markdown back to Drive.

In [ ]:
# Install OCR dependencies. Run once per Colab session.
!pip install -q "surya-ocr==0.17.1" "transformers>=4.56.1,<5" "pypdf>=6,<7"

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Check if GPU available

In [ ]:
import torch
print(torch.cuda.is_available())

Check the input path data

In [ ]:
# Folder in Google Drive where raw PDFs are stored.
from pathlib import Path
MOUNT_PATH = Path("/content/drive/MyDrive/vidyalaya/books/scert_class_8")
if MOUNT_PATH.exists() and MOUNT_PATH.is_dir():
    print(f"Files in {MOUNT_PATH}:")
    files = [f.name for f in MOUNT_PATH.iterdir() if f.is_file()]
    for file in sorted(files):
        print(f"- {file}")
else:
    print(f"Directory not found: {MOUNT_PATH}")

## Surya Runtime Settings

These are intentionally conservative for a Colab T4. Increase only after a book runs successfully.

In [ ]:
import os

os.environ["TORCH_DEVICE"] = "cuda"
os.environ["DETECTOR_BATCH_SIZE"] = "4"
os.environ["RECOGNITION_BATCH_SIZE"] = "32"
os.environ["LAYOUT_BATCH_SIZE"] = "4"

## Config

Edit this cell for each book. Set `PAGE_BATCH_SIZE` to `10` or `20` depending on Colab stability.

In [ ]:
from pathlib import Path

BOARD = "scert_odisha"
CLASS_NO = 8
SUBJECT = "social_science"
BOOK_NAME = "Social Science"
LANGUAGE = "or"

PDF_NAME = "Social_Science.pdf"
MOUNT_PATH = Path("/content/drive/MyDrive/vidyalaya/books/scert_class_8")
PDF_PATH = MOUNT_PATH / PDF_NAME

PAGE_BATCH_SIZE = 10

# All final OCR outputs stay under this Drive root.
OUTPUT_ROOT = Path("/content/drive/MyDrive/vidyalaya/data/processed/ocr") / BOARD / f"class_{CLASS_NO}"

RAW_OUTPUT_PATH = OUTPUT_ROOT / "raw" / f"{SUBJECT}.json"
JSONL_OUTPUT_PATH = OUTPUT_ROOT / "jsonl" / f"{SUBJECT}.jsonl"
MD_OUTPUT_PATH = OUTPUT_ROOT / "md" / f"{SUBJECT}.md"

RAW_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
JSONL_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
MD_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

print("PDF:", PDF_PATH)
print("Book name:", BOOK_NAME)
print("Language:", LANGUAGE)
print("Page batch size:", PAGE_BATCH_SIZE)
print("Raw JSON:", RAW_OUTPUT_PATH)
print("JSONL:", JSONL_OUTPUT_PATH)
print("Markdown:", MD_OUTPUT_PATH)


## Run Surya OCR in Page Batches

This avoids running Surya on a full PDF at once. Final merged raw OCR is saved as `raw/<subject>.json`.

In [ ]:
import json
import shutil
import subprocess
import time

from pypdf import PdfReader, PdfWriter


def split_pdf_into_batches(input_pdf, batch_dir, page_batch_size):
    reader = PdfReader(input_pdf)
    total_pages = len(reader.pages)
    batch_dir.mkdir(parents=True, exist_ok=True)
    batches = []

    for start_page in range(1, total_pages + 1, page_batch_size):
        end_page = min(start_page + page_batch_size - 1, total_pages)
        output_pdf = batch_dir / f"{input_pdf.stem}_p{start_page:04d}_{end_page:04d}.pdf"

        writer = PdfWriter()
        for page_no in range(start_page, end_page + 1):
            writer.add_page(reader.pages[page_no - 1])

        with output_pdf.open("wb") as file:
            writer.write(file)

        batches.append({
            "pdf": output_pdf,
            "start_page": start_page,
            "end_page": end_page,
        })

    return batches


def load_batch_pages(results_dir, batch_pdf):
    result_files = sorted(results_dir.rglob("results.json"))
    if not result_files:
        raise FileNotFoundError(f"No Surya results.json found under {results_dir}")

    data = json.loads(result_files[0].read_text(encoding="utf-8"))
    pages = data.get(batch_pdf.stem)
    if pages is None and len(data) == 1:
        pages = next(iter(data.values()))
    if not isinstance(pages, list):
        raise ValueError(f"Could not find page results for {batch_pdf.stem}")

    return pages


if not PDF_PATH.exists():
    raise FileNotFoundError(f"PDF not found: {PDF_PATH}")

LOCAL_WORK_ROOT = Path("/content/vidyalaya_ocr_work") / SUBJECT
LOCAL_PDF_PATH = LOCAL_WORK_ROOT / PDF_NAME
BATCH_DIR = LOCAL_WORK_ROOT / "pdf_batches"
SURYA_WORK_DIR = LOCAL_WORK_ROOT / "surya_batches"

if LOCAL_WORK_ROOT.exists():
    shutil.rmtree(LOCAL_WORK_ROOT)
LOCAL_WORK_ROOT.mkdir(parents=True, exist_ok=True)

print("Copying PDF to local runtime:", LOCAL_PDF_PATH)
shutil.copyfile(PDF_PATH, LOCAL_PDF_PATH)

batches = split_pdf_into_batches(LOCAL_PDF_PATH, BATCH_DIR, PAGE_BATCH_SIZE)
print(f"Created {len(batches)} batches of up to {PAGE_BATCH_SIZE} pages")

all_pages = []
start = time.monotonic()

for index, batch in enumerate(batches, start=1):
    batch_pdf = batch["pdf"]
    batch_output_dir = SURYA_WORK_DIR / batch_pdf.stem
    batch_output_dir.mkdir(parents=True, exist_ok=True)

    command = [
        "surya_ocr",
        str(batch_pdf),
        "--output_dir",
        str(batch_output_dir),
    ]

    print(f"[{index}/{len(batches)}] Pages {batch['start_page']}-{batch['end_page']}")
    print("Running:", " ".join(command))
    subprocess.run(command, check=True)

    batch_pages = load_batch_pages(batch_output_dir, batch_pdf)
    for local_index, page in enumerate(batch_pages, start=0):
        page["page"] = batch["start_page"] + local_index
        all_pages.append(page)

elapsed = time.monotonic() - start
print(f"Surya finished {len(all_pages)} pages in {elapsed / 60:.1f} minutes")

merged_raw = {
    PDF_PATH.stem: all_pages,
}
RAW_OUTPUT_PATH.write_text(json.dumps(merged_raw, ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved merged raw JSON:", RAW_OUTPUT_PATH)

## Convert Raw Surya JSON to Page JSONL and Markdown

In [ ]:
import json
import shutil
import subprocess
import time

def load_surya_pages(raw_path, pdf_path):
    data = json.loads(raw_path.read_text(encoding="utf-8"))
    pages = data.get(pdf_path.stem)

    if pages is None and len(data) == 1:
        pages = next(iter(data.values()))
    if not isinstance(pages, list):
        raise ValueError(f"Could not find page results for {pdf_path.stem}")

    return pages


def page_text(page):
    lines = []
    for line in page.get("text_lines", []):
        text = str(line.get("text", "")).strip()
        if text:
            lines.append(text)
    return "\n".join(lines)


def write_jsonl_and_markdown(pages):
    with JSONL_OUTPUT_PATH.open("w", encoding="utf-8") as jsonl_file:
        for index, page in enumerate(pages, start=1):
            page_no = page.get("page", index)
            record = {
                "board": BOARD,
                "class": CLASS_NO,
                "subject": SUBJECT,
                "source_pdf": str(PDF_PATH),
                "page_no": page_no,
                "text": page_text(page),
            }
            jsonl_file.write(json.dumps(record, ensure_ascii=False) + "\n")

    markdown_parts = [
        "---",
        f"board: {BOARD}",
        f"class: {CLASS_NO}",
        f"subject: {SUBJECT}",
        f"source_pdf: {PDF_PATH}",
        "ocr_engine: surya",
        f"page_batch_size: {PAGE_BATCH_SIZE}",
        "---",
        "",
    ]

    for index, page in enumerate(pages, start=1):
        page_no = page.get("page", index)
        markdown_parts.append(f"## Page {page_no}")
        markdown_parts.append("")
        markdown_parts.append(page_text(page))
        markdown_parts.append("")

    MD_OUTPUT_PATH.write_text("\n".join(markdown_parts), encoding="utf-8")


pages = load_surya_pages(RAW_OUTPUT_PATH, PDF_PATH)
write_jsonl_and_markdown(pages)

print("Pages:", len(pages))
print("Saved JSONL:", JSONL_OUTPUT_PATH)
print("Saved Markdown:", MD_OUTPUT_PATH)

## Optional: Rewrite JSONL/Markdown with RAG Metadata

Run this after the config cell and after raw JSON exists. It rewrites the same `JSONL_OUTPUT_PATH` and `MD_OUTPUT_PATH` with `book_name`, `book_id`, `language`, and filename-only `source_pdf`.

In [ ]:
import re


def slugify(value):
    value = str(value).strip().lower()
    value = re.sub(r"[^a-z0-9]+", "_", value)
    return value.strip("_")


BOOK_ID = f"{BOARD}_class_{CLASS_NO}_{SUBJECT}_{slugify(BOOK_NAME)}"
SOURCE_PDF_NAME = PDF_NAME


def write_rag_jsonl_and_markdown(pages):
    with JSONL_OUTPUT_PATH.open("w", encoding="utf-8") as jsonl_file:
        for index, page in enumerate(pages, start=1):
            page_no = page.get("page", index)
            record = {
                "board": BOARD,
                "class": CLASS_NO,
                "subject": SUBJECT,
                "book_name": BOOK_NAME,
                "book_id": BOOK_ID,
                "language": LANGUAGE,
                "source_pdf": SOURCE_PDF_NAME,
                "page_no": page_no,
                "text": page_text(page),
            }
            jsonl_file.write(json.dumps(record, ensure_ascii=False) + "\n")

    markdown_parts = [
        "---",
        f"board: {BOARD}",
        f"class: {CLASS_NO}",
        f"subject: {SUBJECT}",
        f"book_name: {BOOK_NAME}",
        f"book_id: {BOOK_ID}",
        f"language: {LANGUAGE}",
        f"source_pdf: {SOURCE_PDF_NAME}",
        "ocr_engine: surya",
        f"page_batch_size: {PAGE_BATCH_SIZE}",
        "---",
        "",
    ]

    for index, page in enumerate(pages, start=1):
        page_no = page.get("page", index)
        markdown_parts.append(f"## Page {page_no}")
        markdown_parts.append("")
        markdown_parts.append(page_text(page))
        markdown_parts.append("")

    MD_OUTPUT_PATH.write_text("\n".join(markdown_parts), encoding="utf-8")


pages = load_surya_pages(RAW_OUTPUT_PATH, PDF_PATH)
write_rag_jsonl_and_markdown(pages)

print("Book ID:", BOOK_ID)
print("Pages:", len(pages))
print("Rewrote JSONL:", JSONL_OUTPUT_PATH)
print("Rewrote Markdown:", MD_OUTPUT_PATH)


## Quick Preview

In [ ]:
print(MD_OUTPUT_PATH.read_text(encoding="utf-8")[:3000])